### RAG Pipelines - Data Ingestion to Vector DB Pipeline


In [28]:
import os
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [29]:
### Read all the pdf's inside the directory

def process_all_pdfs(pdf_directory):
    """ Process all the PDF files in a directory"""

    all_documents =[]
    pdf_dir = Path(pdf_directory)

    # Find all the pdf files in the directory recursively and return as a list all this done by .rglob() which searches file recursively with the file endign .pdf
    pdf_files = list(pdf_dir.rglob("*.pdf"))

    print(f"Found {len(pdf_files)} PDF files in {pdf_directory}")

    for pdf_file in pdf_files:
        print(f"\nProcessing {pdf_file.name}...")
        try:
            # Load the PDF file using PyMuPDFLoader
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source infomation to metadata
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.append(documents)
            print(f"Loaded {len(documents)} pages from {pdf_file.name}")
                
        except Exception as e:
            print(f"Error processing {pdf_file.name}: {e}")
        
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

all_pdf_documents = process_all_pdfs("../data/pdf")

Found 3 PDF files in ../data/pdf

Processing FINALEPICS_finalreport_1-1.pdf...
Loaded 27 pages from FINALEPICS_finalreport_1-1.pdf

Processing Facto_JD_Founders_Office_Sales_Growth.pdf...
Loaded 3 pages from Facto_JD_Founders_Office_Sales_Growth.pdf

Processing HCSTGAI01052026020.pdf...
Loaded 1 pages from HCSTGAI01052026020.pdf

Total documents loaded: 3


In [30]:
all_pdf_documents

[[Document(metadata={'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-03-31T18:42:33+05:30', 'source': '../data/pdf/FINALEPICS_finalreport_1-1.pdf', 'file_path': '../data/pdf/FINALEPICS_finalreport_1-1.pdf', 'total_pages': 27, 'format': 'PDF 1.7', 'title': '', 'author': 'Un-named', 'subject': '', 'keywords': '', 'moddate': '2026-03-31T18:42:33+05:30', 'trapped': '', 'modDate': "D:20260331184233+05'30'", 'creationDate': "D:20260331184233+05'30'", 'page': 0, 'source_file': 'FINALEPICS_finalreport_1-1.pdf', 'file_type': 'pdf'}, page_content='1 \n \n \nAI-Driven Learning Support System with Content Curation and Smart \nLearning \n \n \nAn Engineering Project in Community Service \n \n \nPhase 2 - Final Report \n \nSubmitted by \nSl. No.       Name                                             Register Number \n1.        Prathamesh M Walunj   23BCE10814 \n                                2.        Anuj Kumar Singh           23BCE10738 \n             

In [31]:
### Text splitting get into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """ Split documents into chunks using RecursiveCharacterTextSplitter"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len
    )
    split_docs =text_splitter.split_documents(documents)  ## It takes the original list of documents and returns a new list conatining smaller Document chunks
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    if split_docs:
        print(f"\n Sample Chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")  # Print first 500 characters of the first chunk
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [32]:
print(type(all_pdf_documents))
print(type(all_pdf_documents[0]))

<class 'list'>
<class 'list'>


In [33]:
### This converts the nested list into a flat list 
all_pdf_documents = [
    doc
    for pdf_documents in all_pdf_documents
    for doc in pdf_documents
]

In [34]:
chunks = split_documents(all_pdf_documents)
chunks

Split 31 documents into 52 chunks

 Sample Chunk:
Content: 1 
 
 
AI-Driven Learning Support System with Content Curation and Smart 
Learning 
 
 
An Engineering Project in Community Service 
 
 
Phase 2 - Final Report 
 
Submitted by 
Sl. No.       Name     ...
Metadata: {'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-03-31T18:42:33+05:30', 'source': '../data/pdf/FINALEPICS_finalreport_1-1.pdf', 'file_path': '../data/pdf/FINALEPICS_finalreport_1-1.pdf', 'total_pages': 27, 'format': 'PDF 1.7', 'title': '', 'author': 'Un-named', 'subject': '', 'keywords': '', 'moddate': '2026-03-31T18:42:33+05:30', 'trapped': '', 'modDate': "D:20260331184233+05'30'", 'creationDate': "D:20260331184233+05'30'", 'page': 0, 'source_file': 'FINALEPICS_finalreport_1-1.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-03-31T18:42:33+05:30', 'source': '../data/pdf/FINALEPICS_finalreport_1-1.pdf', 'file_path': '../data/pdf/FINALEPICS_finalreport_1-1.pdf', 'total_pages': 27, 'format': 'PDF 1.7', 'title': '', 'author': 'Un-named', 'subject': '', 'keywords': '', 'moddate': '2026-03-31T18:42:33+05:30', 'trapped': '', 'modDate': "D:20260331184233+05'30'", 'creationDate': "D:20260331184233+05'30'", 'page': 0, 'source_file': 'FINALEPICS_finalreport_1-1.pdf', 'file_type': 'pdf'}, page_content='1 \n \n \nAI-Driven Learning Support System with Content Curation and Smart \nLearning \n \n \nAn Engineering Project in Community Service \n \n \nPhase 2 - Final Report \n \nSubmitted by \nSl. No.       Name                                             Register Number \n1.        Prathamesh M Walunj   23BCE10814 \n                                2.        Anuj Kumar Singh           23BCE10738 \n              

### Embedding and VectorStoreDB



In [35]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [36]:
class EmbeddingManager:
    """ Handles document embedding generation using SentenceTransformer"""

    def __init__(self,model_name:str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name: Huggingface model name for sentence embeddings.
        """
        self.model_name = model_name
        self.model=None  # self.model is a attribute that will hold the loaded SentenceTransformer model. It is initialized to None, indicating that the model has not been loaded yet.
        self._load_model()
    
    def _load_model(self):
        """Load the sentence transformer model"""
        try:
            print(f"Loading model {self.model_name}...")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model {self.model_name} loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts.

        Args:
            texts: List of strings to embed.
        
        Returns:
            Numpy array of embeddings with shape(len(texts)),embedding_dim.
        """
        if not self.model:
            raise ValueError("Model is not loaded. Call load_model() first.")
        
        try:
            print(f"Generating embeddings for {len(texts)} texts...")
            embeddings = self.model.encode(texts, show_progress_bar=True)
            print(f"Generated embeddings with shape: {embeddings.shape}")
            return embeddings
        except Exception as e:
            print(f"Error generating embeddings: {e}")
            raise
    
    def get_sentence_embedding_dimension(self) -> int:
        """Get the dimension of the sentence embeddings."""
        if not self.model:
            raise ValueError("Model is not loaded. Call load_model() first.")
        return self.model.get_sentence_embedding_dimension()


### initialize the embedding manager
embedding_manager = EmbeddingManager()

embedding_manager

Loading model all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3964.26it/s]


Model all-MiniLM-L6-v2 loaded successfully. Embedding dimension: 384


/tmp/ipykernel_27197/1345579887.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model {self.model_name} loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


### VectorStore

In [37]:
class VectorStore:
    """Manages document embedding in a ChromaDB vector store."""

    def __init__(self,collection_name: str="pdf_documents", persistent_directory: str = "../data/vector_store"):
        """
        Initialize the vector store.

        Args:
            collection_name: Name of the ChromaDB collection.
            persistent_directory: Directory to persist the ChromaDB data.
        """
        self.collection_name = collection_name
        self.persistent_directory = persistent_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize the ChromaDB client and collection."""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persistent_directory, exist_ok=True)
            print(f"Initializing ChromaDB client with persistent directory: {self.persistent_directory}")

            self.client = chromadb.PersistentClient(
                path=self.persistent_directory
            )
            # Get or create the collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description":"PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized with collection '{self.collection_name}' in directory '{self.persistent_directory}'")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing ChromaDB client or collection: {e}")
            raise

    def add_documents(self, documents: List[Dict[str, Any]], embeddings: np.ndarray):
        """
        Add documents to the vector store.

        Args:
            documents: List of dictionaries containing 'content' and 'metadata' or LangChain documents .
            embedding_manager: Instance of EmbeddingManager to generate embeddings.
        """
        

        if len(documents) != len(embeddings):
            raise ValueError("The number of documents and embeddings must match.")

        print(f"Adding {len(documents)} documents to the vector store...") 

        # Prepare data for ChromaDB
        ids=[]  #Unique IDs for each document
        metadatas=[] #Metadata for each document
        documents_texts=[] #Actual Text content of the documents
        embeddings_list=[] #Numerical vectors for each document



        for i, (doc,embedding) in enumerate(zip(documents, embeddings)):
                # Generate unique ID
                doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}" # want only the stating 8 character of uuid
                ids.append(doc_id)

                # Prepare metadata
                metadata = dict(doc.metadata)
                metadata['doc_index'] = i
                metadata['content_length'] = len(doc.page_content)
                metadatas.append(metadata)

                # Document content
                documents_texts.append(doc.page_content)

                # Embedding
                embeddings_list.append(embedding.tolist())  # Convert numpy array to list for ChromaDB
        try:
            # Add to ChromaDB collection
            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                documents=documents_texts,
                embeddings=embeddings_list
            )
        except Exception as e:
            print(f"Error adding documents to ChromaDB collection: {e}")
            raise
vectorstore = VectorStore()
vectorstore

Initializing ChromaDB client with persistent directory: ../data/vector_store
Vector store initialized with collection 'pdf_documents' in directory '../data/vector_store'
Existing documents in collection: 156


In [38]:
chunks

[Document(metadata={'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-03-31T18:42:33+05:30', 'source': '../data/pdf/FINALEPICS_finalreport_1-1.pdf', 'file_path': '../data/pdf/FINALEPICS_finalreport_1-1.pdf', 'total_pages': 27, 'format': 'PDF 1.7', 'title': '', 'author': 'Un-named', 'subject': '', 'keywords': '', 'moddate': '2026-03-31T18:42:33+05:30', 'trapped': '', 'modDate': "D:20260331184233+05'30'", 'creationDate': "D:20260331184233+05'30'", 'page': 0, 'source_file': 'FINALEPICS_finalreport_1-1.pdf', 'file_type': 'pdf'}, page_content='1 \n \n \nAI-Driven Learning Support System with Content Curation and Smart \nLearning \n \n \nAn Engineering Project in Community Service \n \n \nPhase 2 - Final Report \n \nSubmitted by \nSl. No.       Name                                             Register Number \n1.        Prathamesh M Walunj   23BCE10814 \n                                2.        Anuj Kumar Singh           23BCE10738 \n              

In [39]:
### Convert the text to embedding

texts = [doc.page_content for doc in chunks]

#Generate embeddings for the chunks
embeddings = embedding_manager.generate_embeddings(texts)

### Store in the vector store
vectorstore.add_documents(chunks, embeddings)


Generating embeddings for 52 texts...


Batches: 100%|██████████| 2/2 [00:02<00:00,  1.19s/it]

Generated embeddings with shape: (52, 384)
Adding 52 documents to the vector store...


### Retriver Pipeline From VectorStore

In [40]:
class RAGRetriever:
    """ Handles query-based retrieval from the vector store"""

    def __init__(self,vector_store: VectorStore,embedding_manager: EmbeddingManager):
        """
        Initialize the retriever

        Args:
            vector_store: Vector store containing embedding
            embedding_manager: Manager for generating query embedding
        """
        self.vector_store = vector_store
        self.embedding_manager=embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float=0.0)-> List[Dict[str,Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top resultss to return
            score_threshold: Minimum similarity score threshold

        Returns:
            List of dictionaries containing retrieved documents and metadata
        """

        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold:{score_threshold}")

        #Generate query embedding

        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            # Process results
            retrieved_docs=[]

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i,(doc_id,document,metadata,distance) in enumerate(zip(ids,documents,metadatas,distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1-distance
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id':doc_id,
                            'content':document,
                            'metadata':metadata,
                            'similarity_score':similarity_score,
                            'distance':distance,
                            'rank':i+1
                        })
                print(f"Retrived {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")

            return retrieved_docs
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return [] 
        
        

rag_retriever=RAGRetriever(vectorstore,embedding_manager)
rag_retriever

In [41]:
rag_retriever.retrieve("Give me the Job Description of Facto")

Retrieving documents for query: 'Give me the Job Description of Facto'
Top K: 5, Score threshold:0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 55.21it/s]

Generated embeddings with shape: (1, 384)
Retrived 5 documents (after filtering)


[{'id': 'doc_a0923905_42',
  'content': "FOR THE TRUE INDUSTRIALIST. \nFounder's Office – Sales & Growth \nInternship with a pre-placement offer path \nWork with both founders. Run real deals with factory owners. Earn a full-time offer at ₹9.6 \nLPA. \nInternship  ·  India first, US as you grow  ·  Works directly with the founders  ·  Location: Remote  ·  Full-time \nuntil you graduate \nMany Indian factories turning over ₹10 Cr to ₹400+ Cr still run on paper registers, WhatsApp groups and Excel. \nThe usual fix is an ERP that takes 18 months to roll out. Facto gets a factory live in 8 to 12 weeks, and more \nthan 50 manufacturers already run on it. \nABOUT FACTO \nFacto (Neev Facto Technologies) builds operations software for mid-sized manufacturers. We sit with a \nfactory, map how it really runs (orders, production, inventory, dispatch, payments) and give the owner \none system that runs all of it. Our own team builds and configures it, and customers go live in 8–12 weeks \ninstead 

### Integration Vectordb Context pipeline With LLM Output

In [ ]:
from IPython.core import magic_arguments
### Simple RAG pipeline with grok LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv 
load_dotenv()

### Initialse the Groq LLM 
groq_api_key=os.getenv("groq_api_key")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="openai/gpt-oss-20b",temperature=0.1,max_tokens=1024)

# 2. Simple RAG function retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else "" 
    if not context:
        return "No relevant context found to answer the question."

    ##Generate the answer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context
        {context}

        Question:{query}
        Answer:
        
        """
    response=llm.invoke(prompt)
    return response.content

In [43]:
answer = rag_simple("Give me the Job Description of Facto?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'Give me the Job Description of Facto?'
Top K: 3, Score threshold:0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 51.38it/s]

Generated embeddings with shape: (1, 384)
Retrived 3 documents (after filtering)


**Facto – Founder's Office (Sales & Growth) Internship**

- **Role**: Sales & Growth Intern in the Founder's Office  
- **Location**: Remote (India‑first, US expansion later)  
- **Duration**: Full‑time until graduation, with a pre‑placement offer path  
- **Compensation**: Full‑time offer at ₹9.6 LPA upon successful completion  

**Key Responsibilities**

1. **Deal Execution**  
   - Work directly with both founders to identify, negotiate, and close deals with mid‑size manufacturing clients.  
   - Manage end‑to‑end sales cycles, from prospecting to contract signing.  

2. **Client Engagement**  
   - Conduct on‑site (or virtual) visits to factory owners, understand their operations (orders, production, inventory, dispatch, payments).  
   - Map current processes and demonstrate how Facto’s unified operations software can replace paper, WhatsApp, and Excel workflows.  

3. **Product & Implementation Support**  
   - Collaborate with the product team to tailor solutions to client needs

### Enchanced RAG Pipeline Features

In [44]:
# ---Enchanced RAG Pipeline Features ---
def rag_advanced(query,retriever,llm,top_k=5,min_score=0.2,return_context=False):
    """ 
    RAG pipeline with extra features:
    -Returns answer,sources,confidence score,and optionally full context.
    """

    results=retriever.retrieve(query,top_k=top_k,score_threshold=min_score)
    if not results:
        return {'answer':'No relevant context found','sources':[],'confidence':0.0,'context':''}

    #Prepare context and sources
    context="\n\n".join([doc['content'] for doc in results])
    sources=[{
        'source':doc['metadata'].get('source_file',doc['metadata'].get('source','unknown')),
        'page':doc['metadata'].get('page','unknown'),
        'score':doc['similarity_score'],
        'preview':doc['content'][:300]+ '...'
    } for doc in results]

    confidence = max([doc['similarity_score'] for doc in results])

    # Generate answer
    prompt=f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuery:{query}"""
    response =llm.invoke([prompt.format(context=context,query=query)])

    output={
        'answer':response.content,
        'sources':sources,
        'confidence':confidence
    }
    if return_context:
        output['context']=context
    return output

# Example usage
result=rag_advanced("Give me the Job Description of Facto?",rag_retriever,llm,top_k=3,min_score=0.0,return_context=True)
print("Answer:", result['answer'])
print("Sources:",result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:",result['context'][:300])

Retrieving documents for query: 'Give me the Job Description of Facto?'
Top K: 3, Score threshold:0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 56.16it/s]

Generated embeddings with shape: (1, 384)
Retrived 3 documents (after filtering)


Answer: **Job Title:**  
Founder's Office – Sales & Growth Intern (Pre‑Placement Offer Path)

**Company:**  
Facto (Neev Facto Technologies) – Operations software for mid‑size manufacturers

**Location:**  
Remote (India first, US expansion later)

**Duration:**  
Full‑time internship until graduation, with a pre‑placement offer at ₹9.6 LPA

---

### Key Responsibilities
- **Direct Collaboration:** Work side‑by‑side with both founders on strategy and execution.  
- **Deal Execution:** Lead end‑to‑end sales cycles with factory owners (₹10 Cr–₹400+ Cr turnover).  
- **Customer Onboarding:** Map factory processes (orders, production, inventory, dispatch, payments) and help configure Facto’s unified system.  
- **Go‑Live Support:** Ensure factories go live in 8–12 weeks, replacing legacy paper/WhatsApp/Excel workflows.  
- **Feedback Loop:** Gather user insights to refine product roadmap and sales messaging.  
- **Growth Initiatives:** Identify and pursue new market segments, partnerships,